In [8]:
# https://datajar.co.uk/
# https://scriptable.app/


from datetime import datetime, timezone, timedelta 
import boto3, os, json

year = os.environ.get("F1_YEAR", "2026")
BUCKET = os.environ.get("DATA_BUCKET")
DATA_SOURCE = os.environ.get("DATA_SOURCE", "auto").lower()
s3 = boto3.client('s3')
print("YEAR =", year)
print("DATA_SOURCE =", DATA_SOURCE)
print("BUCKET =", BUCKET)

def load_json_from_s3(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return json.loads(obj['Body'].read())

def load_json_from_local(key):
    with open(key, 'r') as file:
        return json.load(file)

def load_inputs(year):
    keys = {
        "sessions": f"{year}_schedule.json",
        "teams": f"{year}_teams.json",
        "drivers": f"{year}_drivers.json"
    }

    use_s3 = DATA_SOURCE == "s3" or (DATA_SOURCE == "auto" and BUCKET)
    if use_s3:
        if not BUCKET:
            raise ValueError("DATA_BUCKET is required when DATA_SOURCE is s3")
        return (
            load_json_from_s3(keys["sessions"]),
            load_json_from_s3(keys["teams"]),
            load_json_from_s3(keys["drivers"]),
        )

    return (
        load_json_from_local(keys["sessions"]),
        load_json_from_local(keys["teams"]),
        load_json_from_local(keys["drivers"]),
    )

def td():
    return -7

def get_next(ss):
    now = datetime.now(timezone.utc)
    for session in ss:
        try:
            start = datetime.strptime(session['start'], '%Y-%m-%dT%H:%M:%S%z')
        except (KeyError, ValueError):
            continue
        if now - timedelta(hours=2) < start:
            return session
    return None



def duration(tdelta):
    d = {"D": tdelta.days}
    d["H"], rem = divmod(tdelta.seconds, 3600)
    d["M"], d["S"] = divmod(rem, 60)
    if d['D'] > 0:
        return f"{d['D']}d {d['H']}h"
    elif d['H'] > 0:
        return f"{d['H']}h {d['M']}m"
    else:
        return f"{d['M']}m"

def delta(start):
    now = datetime.now(timezone.utc)
    if now < start:
            return f"{duration(start - now)}"
    return  "Live"

def dow(start, offset):
    return (start + timedelta(hours=offset)).strftime('%a')

def dom(start, offset):
    return (start + timedelta(hours=offset)).strftime('%-d')


sessions, teams, drivers = load_inputs(year)


#   Runtime here
next = get_next(sessions)
if not next:
    print(json.dumps({"error": "No upcoming session found"}))
else:
    start = datetime.strptime(next['start'], '%Y-%m-%dT%H:%M:%S%z')
    # dates
    next['dow'] = dow(start, td())
    next['dom'] = dom(start, td())
    next['delta'] = delta(start)
    for t in teams:
        next[t['team_name'].lower()] = t['place']
    for d in drivers:
        abr = d['first_name'].lower()[0] + d['last_name'].lower()[0] + str(d['car_number'])
        next[abr] = d['place']

    print(json.dumps(next))



BUCKET = f1-data-00000000
{"event": "Sakhir", "session": "Winter", "start": "2026-02-11T09:30:00-00:00", "dow": "Wed", "dom": "11", "delta": "53d 7h", "mclaren": "1", "mercedes": "2", "red bull racing": "3", "ferrari": "4", "williams": "5", "racing bulls": "6", "aston martin": "7", "haas f1 team": "8", "kick sauber": "9", "alpine": "10", "ln4": "1", "mv3": "2", "op81": "3", "gr63": "4", "cl16": "5", "lh44": "6", "ka12": "7", "aa23": "8", "cs55": "9", "fa14": "10", "nh27": "11", "ih6": "12", "ob87": "13", "ll30": "14", "eo31": "15", "ls18": "16", "yt22": "17", "pg10": "18", "gb5": "19", "fc43": "20", "jd7": "21"}
